# NSF Awards Cleaning and Preparation

## GrantScopeAI — Notebook 03

### Purpose

This notebook cleans and prepares the raw National Science Foundation awards data for integration into GrantScopeAI.

The NSF extraction was built from multiple AI–chemistry and materials-related search queries. Because one award may match more than one query, the raw dataset can contain repeated award IDs. These repeated records must be investigated and consolidated without losing information about which queries produced each match.

### Main objectives

This notebook will:

- load and inspect the raw NSF awards dataset;
- assess missing values, data types, and field coverage;
- investigate duplicate award IDs caused by query overlap;
- consolidate the data to one row per NSF award;
- preserve matched-query and extraction lineage;
- standardise titles, abstracts, dates, programmes, organisations, and USD amounts;
- create fields that align with the GrantScopeAI common grant schema;
- apply the 2021–2025 project scope;
- create broad and refined AI–chemistry/materials relevance tiers;
- validate and export the cleaned NSF datasets.

### Completion criteria

This notebook is complete when:

1. every retained NSF award has one unique award ID;
2. duplicate-query matches have been consolidated without losing lineage;
3. important dates and monetary fields use appropriate data types;
4. each award has a standardised source key, currency, organisation, and URL;
5. relevance rules have been reviewed using sampled records;
6. the full interim and candidate datasets have been validated and saved.

## 1. Load the raw NSF awards data

The NSF awards extraction contains records collected from the NSF Awards API for the 2021–2025 project period.

This section:

- identifies the most recent raw NSF extraction file;
- loads all columns as strings to preserve the original API values;
- creates a working copy for cleaning and transformation;
- confirms the number of rows and columns loaded.

The raw DataFrame is preserved separately so that all cleaning decisions can be traced back to the original extracted data.

In [1]:
from pathlib import Path
import pandas as pd

# Project folders
PROJECT_ROOT = Path(
    r"C:\Users\kahau\OneDrive\Documents\Techmeup"
    r"\Final_Project\GrantScopeAI"
)

NSF_RAW_DIR = (
    PROJECT_ROOT
    / "Data"
    / "Raw_Data"
)

# Locate the NSF raw extraction
nsf_raw_files = sorted(
    NSF_RAW_DIR.glob("nsf_grantscope_raw_*.csv")
)

if not nsf_raw_files:
    raise FileNotFoundError(
        f"No NSF raw CSV was found in: {NSF_RAW_DIR}"
    )

# Use the most recently named extraction
nsf_raw_path = nsf_raw_files[-1]

# Load the source data
nsf_raw_df = pd.read_csv(
    nsf_raw_path,
    dtype="string",
    low_memory=False
)

# Work on a separate copy so the raw import remains unchanged
nsf_df = nsf_raw_df.copy()

print("NSF file loaded:")
print(nsf_raw_path)

print(
    "\nRaw rows × columns:",
    nsf_raw_df.shape
)

print(
    "Working-copy rows × columns:",
    nsf_df.shape
)

NSF file loaded:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Raw_Data\nsf_grantscope_raw_2021_2025_2026-08-01.csv

Raw rows × columns: (33125, 70)
Working-copy rows × columns: (33125, 70)


In [2]:
print(
    "Total columns:",
    len(nsf_df.columns)
)

print("\nNSF columns:\n")

for column_number, column_name in enumerate(
    nsf_df.columns,
    start=1
):
    print(
        f"{column_number:02d}. {column_name}"
    )

print("\nFirst two records, transposed:")

display(
    nsf_df
    .head(2)
    .transpose()
    .rename(
        columns={
            0: "record_1",
            1: "record_2"
        }
    )
)

Total columns: 70

NSF columns:

01. abstractText
02. activeAwd
03. agency
04. awardAgencyCode
05. awardee
06. awardeeAddress
07. awardeeCity
08. awardeeCountryCode
09. awardeeDistrict
10. awardeeDistrictCode
11. awardeeName
12. awardeePhone
13. awardeeStateCode
14. awardeeZipCode
15. cfdaNumber
16. date
17. dirAbbr
18. divAbbr
19. estimatedTotalAmt
20. expDate
21. fundAgencyCode
22. fundProgramName
23. fundsObligated
24. fundsObligatedAmt
25. histAwd
26. id
27. initAmendmentDate
28. jrnl
29. latestAmendmentDate
30. managingPec
31. orgCodeDir
32. orgCodeDiv
33. orgLongName
34. orgLongName2
35. orgUrl
36. parentUeiNumber
37. pdPIName
38. perfAddress
39. perfCity
40. perfCountryCode
41. perfDistrict
42. perfDistrictCode
43. perfLocation
44. perfStateCode
45. perfZipCode
46. pi
47. piEmail
48. piFirstName
49. piId
50. piLastName
51. poEmail
52. poName
53. poPhone
54. primaryProgram
55. progEleCode
56. program
57. progRefCode
58. projectOutComesReport
59. publicAccessMandate
60. publicatio

,record_1,record_2
abstractText,Composite structures have increasingly emerged...,National efforts to digitize natural history c...
activeAwd,false,false
agency,NSF,NSF
awardAgencyCode,4900,4900
awardee,VIRGINIA POLYTECHNIC INSTITUTE & STATE UNIVERSITY,UNIVERSITY OF FLORIDA
...,...,...
piMiddeInitial,<NA>,P
coPDPI,<NA>,<NA>
awdSpAttnCode,<NA>,<NA>
awdSpAttnDesc,<NA>,<NA>


In [3]:
core_nsf_columns = [
    "id",
    "title",
    "abstractText",
    "startDate",
    "expDate",
    "fundsObligatedAmt",
    "estimatedTotalAmt",
    "fundProgramName",
    "primaryProgram",
    "dirAbbr",
    "divAbbr",
    "awardeeName",
    "awardeeCity",
    "awardeeStateCode",
    "awardeeCountryCode",
    "_matched_query"
]

core_field_summary = []

for column in core_nsf_columns:
    non_missing_values = nsf_df[column].dropna()

    examples = (
        non_missing_values
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .head(3)
        .tolist()
    )

    core_field_summary.append({
        "column": column,
        "dtype": str(nsf_df[column].dtype),
        "missing_rows": nsf_df[column].isna().sum(),
        "missing_percent": round(
            nsf_df[column].isna().mean() * 100,
            2
        ),
        "unique_values": nsf_df[column].nunique(dropna=True),
        "examples": " | ".join(examples)
    })

core_field_summary_df = pd.DataFrame(core_field_summary)

display(core_field_summary_df)

,column,dtype,missing_rows,missing_percent,unique_values,examples
0,id,string,0,0.00,19916,2035038 | 2027234 | 2108526
1,title,string,0,0.00,16295,Ultra-high Precision Assembly of Aerospace Com...
2,abstractText,string,6,0.02,16414,Composite structures have increasingly emerged...
3,startDate,string,0,0.00,121,01/15/2021 | 01/15/2021 | 01/15/2021
4,expDate,string,0,0.00,146,12/31/2024 | 12/31/2024 | 12/31/2022
5,fundsObligatedAmt,string,0,0.00,13555,319386 | 292392 | 200000
6,estimatedTotalAmt,string,0,0.00,12077,319386 | 292392 | 200000
7,fundProgramName,string,49,0.15,3215,AM-Advanced Manufacturing | Capacity: Cyberinf...
8,primaryProgram,string,0,0.00,1687,['01002122DB NSF RESEARCH & RELATED ACTIVIT'] ...
9,dirAbbr,string,0,0.00,9,ENG | BIO | BIO


In [4]:
award_id_counts_df = (
    nsf_df["id"]
    .value_counts()
    .rename_axis("award_id")
    .reset_index(name="row_count")
)

repeated_award_ids = award_id_counts_df.loc[
    award_id_counts_df["row_count"].gt(1),
    "award_id"
]

duplicate_award_rows_df = (
    nsf_df.loc[
        nsf_df["id"].isin(repeated_award_ids)
    ]
    .copy()
)

print("Total raw rows:", f"{len(nsf_df):,}")

print(
    "Unique award IDs:",
    f"{nsf_df['id'].nunique():,}"
)

print(
    "Extra rows caused by repeated award IDs:",
    f"{len(nsf_df) - nsf_df['id'].nunique():,}"
)

print(
    "Award IDs appearing more than once:",
    f"{len(repeated_award_ids):,}"
)

print(
    "Maximum rows for one award ID:",
    int(award_id_counts_df["row_count"].max())
)

print("\nDistribution of rows per award:")

display(
    award_id_counts_df["row_count"]
    .value_counts()
    .sort_index()
    .rename_axis("rows_per_award")
    .reset_index(name="award_count")
)

print("\nExample repeated awards:")

display(
    duplicate_award_rows_df[
        [
            "id",
            "title",
            "_matched_query",
            "fundsObligatedAmt",
            "awardeeName"
        ]
    ]
    .sort_values(["id", "_matched_query"])
    .head(20)
)

Total raw rows: 33,125
Unique award IDs: 19,916
Extra rows caused by repeated award IDs: 13,209
Award IDs appearing more than once: 8,503
Maximum rows for one award ID: 9

Distribution of rows per award:


,rows_per_award,award_count
0,1,11413
1,2,5377
2,3,2063
3,4,707
4,5,234
5,6,93
6,7,20
7,8,8
8,9,1



Example repeated awards:


,id,title,_matched_query,fundsObligatedAmt,awardeeName
26339,1906383,Enabling Quantum Leap: Q-AMASE-i: MonArk Quant...,data-driven AND materials,22222678,Montana State University
1010,1906383,Enabling Quantum Leap: Q-AMASE-i: MonArk Quant...,machine AND learning,22222678,Montana State University
16328,1910284,III:Small: Counterfactually Fair Machine Learn...,deep AND learning,484828,University of Arkansas
1541,1910284,III:Small: Counterfactually Fair Machine Learn...,machine AND learning,484828,University of Arkansas
1569,1910284,III:Small: Counterfactually Fair Machine Learn...,machine AND learning,484828,University of Arkansas
22660,1936438,Collaborative Research: Impacting Assessment P...,data-driven AND chemistry,12540,University of Wisconsin-Milwaukee
25693,1936438,Collaborative Research: Impacting Assessment P...,data-driven AND materials,12540,University of Wisconsin-Milwaukee
22652,1936517,Collaborative Research: Impacting Assessment P...,data-driven AND chemistry,34172,Grand Valley State University
25680,1936517,Collaborative Research: Impacting Assessment P...,data-driven AND materials,34172,Grand Valley State University
22654,1936574,Collaborative Research: Impacting Assessment P...,data-driven AND chemistry,251315,University of South Florida


In [5]:
# Check whether repeated award rows differ in fields other than
# the search query that returned them.

columns_to_compare = [
    column
    for column in nsf_df.columns
    if column != "_matched_query"
]

duplicate_field_conflicts_df = (
    duplicate_award_rows_df
    .groupby("id", dropna=False)[columns_to_compare]
    .nunique(dropna=False)
    .gt(1)
    .sum()
    .sort_values(ascending=False)
    .rename("award_ids_with_different_values")
    .reset_index()
    .rename(columns={"index": "column"})
)

conflicting_fields_df = (
    duplicate_field_conflicts_df.loc[
        duplicate_field_conflicts_df[
            "award_ids_with_different_values"
        ].gt(0)
    ]
    .reset_index(drop=True)
)

print(
    "Repeated award IDs checked:",
    f"{duplicate_award_rows_df['id'].nunique():,}"
)

print(
    "Fields with differences across repeated rows:",
    f"{len(conflicting_fields_df):,}"
)

display(conflicting_fields_df)

Repeated award IDs checked: 8,503
Fields with differences across repeated rows: 0


,column,award_ids_with_different_values


## Consolidating duplicate NSF award records

The raw extraction contains more rows than unique NSF award IDs because the same award can be returned by multiple search queries or repeated retrieval windows.

The duplicate investigation confirmed that:

- repeated rows with the same award ID contain identical award information;
- the only meaningful difference is the query that retrieved the award;
- some awards match several distinct search queries;
- some award–query combinations were retrieved more than once.

These repeated rows therefore do not represent separate grants.

The following step consolidates the dataset to one row per NSF award ID. All unique matched queries are retained in a combined lineage field, along with the number of distinct queries associated with each award. This preserves extraction provenance while preventing duplicate awards from affecting later counts and analysis.

In [6]:
def join_unique_queries(values):
    unique_queries = (
        values
        .dropna()
        .astype(str)
        .str.strip()
    )

    unique_queries = unique_queries[
        unique_queries.ne("")
    ].unique()

    return " | ".join(sorted(unique_queries))


# Create one query-lineage record per award
nsf_query_lineage_df = (
    nsf_df
    .groupby("id", dropna=False)
    .agg(
        matched_queries=(
            "_matched_query",
            join_unique_queries
        ),
        matched_query_count=(
            "_matched_query",
            "nunique"
        )
    )
    .reset_index()
)

# Retain one unchanged award record per ID
nsf_unique_awards_df = (
    nsf_df
    .drop(columns="_matched_query")
    .drop_duplicates(
        subset="id",
        keep="first"
    )
    .merge(
        nsf_query_lineage_df,
        on="id",
        how="left",
        validate="one_to_one"
    )
    .reset_index(drop=True)
)

nsf_unique_awards_df["has_query_overlap"] = (
    nsf_unique_awards_df["matched_query_count"].gt(1)
)

print(
    "Raw rows:",
    f"{len(nsf_df):,}"
)

print(
    "Consolidated award rows:",
    f"{len(nsf_unique_awards_df):,}"
)

print(
    "Unique award IDs:",
    f"{nsf_unique_awards_df['id'].nunique():,}"
)

print(
    "Duplicate award IDs remaining:",
    nsf_unique_awards_df["id"].duplicated().sum()
)

print(
    "Awards matched by multiple queries:",
    f"{nsf_unique_awards_df['has_query_overlap'].sum():,}"
)

display(
    nsf_unique_awards_df[
        [
            "id",
            "title",
            "matched_query_count",
            "matched_queries"
        ]
    ]
    .sort_values(
        "matched_query_count",
        ascending=False
    )
    .head(10)
)

Raw rows: 33,125
Consolidated award rows: 19,916
Unique award IDs: 19,916
Duplicate award IDs remaining: 0
Awards matched by multiple queries: 7,139


,id,title,matched_query_count,matched_queries
8271,2133650,III:Small: Interpretable Deep Generative Model...,9,artificial AND intelligence | cheminformatics ...
736,2100971,"Catalyst: Chemiluminescence, Quantum Yield and...",8,artificial AND intelligence | computational AN...
3303,2229755,RII Track-4: NSF: Data-driven Computational an...,8,artificial AND intelligence | cheminformatics ...
1930,2150191,REU Site: Advancing high-performance computing...,7,artificial AND intelligence | computational AN...
1355,2112356,Category II: ACES - Accelerating Computing for...,7,artificial AND intelligence | computational AN...
1460,2119672,Collaborative Research: DMREF: Accelerated Des...,7,artificial AND intelligence | computational AN...
442,2043205,CAREER: Guiding automated synthesis of hybrid ...,7,artificial AND intelligence | autonomous AND l...
348,2053195,Hydration Structures and Perturbed Hydrogen Bo...,7,artificial AND intelligence | computational AN...
3453,2311104,CDS&E: A Deep Learning Framework for Evaluatio...,7,artificial AND intelligence | computational AN...
1077,2141384,EAGER: ADAPT: AI Guided Design and Synthesis o...,7,artificial AND intelligence | computational AN...


In [7]:
award_query_pair_counts_df = (
    nsf_df
    .groupby(
        ["id", "_matched_query"],
        dropna=False
    )
    .size()
    .reset_index(name="row_count")
)

repeated_same_query_pairs_df = (
    award_query_pair_counts_df.loc[
        award_query_pair_counts_df["row_count"].gt(1)
    ]
    .copy()
)

awards_repeated_with_same_query = (
    repeated_same_query_pairs_df["id"].nunique()
)

awards_with_multiple_unique_queries = (
    nsf_query_lineage_df["matched_query_count"].gt(1).sum()
)

awards_with_repeated_rows = (
    award_id_counts_df["row_count"].gt(1).sum()
)

print(
    "Awards with repeated raw rows:",
    f"{awards_with_repeated_rows:,}"
)

print(
    "Awards matching multiple unique queries:",
    f"{awards_with_multiple_unique_queries:,}"
)

print(
    "Awards repeated at least once under the same query:",
    f"{awards_repeated_with_same_query:,}"
)

print(
    "Repeated award–query combinations:",
    f"{len(repeated_same_query_pairs_df):,}"
)

print(
    "Maximum repetitions of one award–query pair:",
    int(repeated_same_query_pairs_df["row_count"].max())
)

display(
    repeated_same_query_pairs_df
    .sort_values("row_count", ascending=False)
    .head(20)
)

Awards with repeated raw rows: 8,503
Awards matching multiple unique queries: 7,139
Awards repeated at least once under the same query: 2,877
Repeated award–query combinations: 3,090
Maximum repetitions of one award–query pair: 2


,id,_matched_query,row_count
30030,2625881,machine AND learning,2
3,1910284,machine AND learning,2
30,1954773,data-driven AND chemistry,2
38,2002635,artificial AND intelligence,2
57,2005749,machine AND learning,2
60,2006456,data-driven AND materials,2
64,2008011,machine AND learning,2
68,2009288,data-driven AND materials,2
95,2013062,machine AND learning,2
98,2013696,data-driven AND materials,2


In [8]:
# Begin the cleaned one-row-per-award working table
nsf_clean_df = nsf_unique_awards_df.copy()

# Preserve the original NSF ID while creating a standardised join key
nsf_clean_df["award_id_clean"] = (
    nsf_clean_df["id"]
    .astype("string")
    .str.strip()
    .str.strip('"')
    .str.strip("'")
    .str.replace(r"\.0$", "", regex=True)
)

# Create the common GrantScopeAI identifiers
nsf_clean_df["grant_key"] = (
    "NSF_" + nsf_clean_df["award_id_clean"]
)

nsf_clean_df["source"] = "NSF"
nsf_clean_df["currency"] = "USD"

print("Cleaned award rows:", f"{len(nsf_clean_df):,}")

print(
    "Unique cleaned award IDs:",
    f"{nsf_clean_df['award_id_clean'].nunique():,}"
)

print(
    "Duplicate cleaned award IDs:",
    nsf_clean_df["award_id_clean"].duplicated().sum()
)

print(
    "Duplicate grant keys:",
    nsf_clean_df["grant_key"].duplicated().sum()
)

print(
    "Missing cleaned award IDs:",
    nsf_clean_df["award_id_clean"].isna().sum()
)

display(
    nsf_clean_df[
        [
            "id",
            "award_id_clean",
            "grant_key",
            "source",
            "currency"
        ]
    ].head()
)

Cleaned award rows: 19,916
Unique cleaned award IDs: 19,916
Duplicate cleaned award IDs: 0
Duplicate grant keys: 0
Missing cleaned award IDs: 0


,id,award_id_clean,grant_key,source,currency
0,2035038,2035038,NSF_2035038,NSF,USD
1,2027234,2027234,NSF_2027234,NSF,USD
2,2108526,2108526,NSF_2108526,NSF,USD
3,2043611,2043611,NSF_2043611,NSF,USD
4,2054691,2054691,NSF_2054691,NSF,USD


In [9]:
text_columns = {
    "title": "title_clean",
    "abstractText": "abstract_clean"
}

for raw_column, clean_column in text_columns.items():
    nsf_clean_df[clean_column] = (
        nsf_clean_df[raw_column]
        .fillna("")
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Preserve clearly named raw versions for the common schema
nsf_clean_df["title_raw"] = nsf_clean_df["title"]
nsf_clean_df["abstract_raw"] = nsf_clean_df["abstractText"]

print(
    "Empty cleaned titles:",
    nsf_clean_df["title_clean"].eq("").sum()
)

print(
    "Empty cleaned abstracts:",
    nsf_clean_df["abstract_clean"].eq("").sum()
)

print(
    "Median title length:",
    int(nsf_clean_df["title_clean"].str.len().median())
)

print(
    "Median abstract length:",
    int(nsf_clean_df["abstract_clean"].str.len().median())
)

display(
    nsf_clean_df[
        [
            "award_id_clean",
            "title_raw",
            "title_clean",
            "abstract_clean"
        ]
    ].head()
)

Empty cleaned titles: 0
Empty cleaned abstracts: 4
Median title length: 100
Median abstract length: 2885


,award_id_clean,title_raw,title_clean,abstract_clean
0,2035038,Ultra-high Precision Assembly of Aerospace Com...,Ultra-high Precision Assembly of Aerospace Com...,Composite structures have increasingly emerged...
1,2027234,Collaborative Research: CIBR: Leaping the Spec...,Collaborative Research: CIBR: Leaping the Spec...,National efforts to digitize natural history c...
2,2108526,RAPID: Real-time Forecasting of COVID-19 risk ...,RAPID: Real-time Forecasting of COVID-19 risk ...,In an effort to support decision making by gov...
3,2043611,SCC-CIVIC-PG Track A: Leveraging AI-assist Mic...,SCC-CIVIC-PG Track A: Leveraging AI-assist Mic...,COVID-19 disproportionately affects the low-wa...
4,2054691,I-Corps: AI-enabled automation intelligence s...,I-Corps: AI-enabled automation intelligence so...,The broader impact/commercial potential of thi...


In [10]:
nsf_date_columns = [
    "startDate",
    "expDate",
    "date",
    "initAmendmentDate",
    "latestAmendmentDate"
]

date_field_summary = []

for column in nsf_date_columns:
    non_missing_values = (
        nsf_clean_df[column]
        .dropna()
        .astype(str)
        .str.strip()
    )

    date_field_summary.append({
        "column": column,
        "dtype": str(nsf_clean_df[column].dtype),
        "missing_rows": nsf_clean_df[column].isna().sum(),
        "unique_values": nsf_clean_df[column].nunique(dropna=True),
        "examples": " | ".join(
            non_missing_values.head(5).tolist()
        )
    })

date_field_summary_df = pd.DataFrame(date_field_summary)

display(date_field_summary_df)

,column,dtype,missing_rows,unique_values,examples
0,startDate,string,0,121,01/15/2021 | 01/15/2021 | 01/15/2021 | 01/15/2...
1,expDate,string,0,146,12/31/2024 | 12/31/2024 | 12/31/2022 | 12/31/2...
2,date,string,0,1329,01/07/2021 | 01/13/2021 | 12/09/2020 | 01/08/2...
3,initAmendmentDate,string,0,1329,01/07/2021 | 01/13/2021 | 12/09/2020 | 01/08/2...
4,latestAmendmentDate,string,0,1445,08/03/2023 | 05/19/2021 | 05/21/2021 | 05/20/2...


In [11]:
for column in nsf_date_columns:
    nsf_clean_df[column] = pd.to_datetime(
        nsf_clean_df[column],
        format="%m/%d/%Y",
        errors="coerce"
    )

nsf_clean_df["award_year"] = (
    nsf_clean_df["startDate"].dt.year
)

for column in nsf_date_columns:
    print(
        f"{column}: "
        f"{nsf_clean_df[column].isna().sum():,} missing/invalid | "
        f"dtype = {nsf_clean_df[column].dtype}"
    )

print(
    "\nMissing award years:",
    nsf_clean_df["award_year"].isna().sum()
)

print("\nAwards by start year:")

display(
    nsf_clean_df["award_year"]
    .value_counts()
    .sort_index()
    .rename_axis("award_year")
    .reset_index(name="award_count")
)

startDate: 0 missing/invalid | dtype = datetime64[ns]
expDate: 0 missing/invalid | dtype = datetime64[ns]
date: 0 missing/invalid | dtype = datetime64[ns]
initAmendmentDate: 0 missing/invalid | dtype = datetime64[ns]
latestAmendmentDate: 0 missing/invalid | dtype = datetime64[ns]

Missing award years: 0

Awards by start year:


,award_year,award_count
0,2021,4478
1,2022,4164
2,2023,4054
3,2024,3798
4,2025,3422


In [12]:
nsf_amount_columns = [
    "fundsObligatedAmt",
    "estimatedTotalAmt",
    "fundsObligated",
    "arraAmount"
]

amount_field_summary = []

for column in nsf_amount_columns:
    non_missing_values = (
        nsf_clean_df[column]
        .dropna()
        .astype(str)
        .str.strip()
    )

    amount_field_summary.append({
        "column": column,
        "dtype": str(nsf_clean_df[column].dtype),
        "missing_rows": nsf_clean_df[column].isna().sum(),
        "unique_values": nsf_clean_df[column].nunique(dropna=True),
        "examples": " | ".join(
            non_missing_values.head(8).tolist()
        )
    })

amount_field_summary_df = pd.DataFrame(
    amount_field_summary
)

display(amount_field_summary_df)

,column,dtype,missing_rows,unique_values,examples
0,fundsObligatedAmt,string,0,13555,319386 | 292392 | 200000 | 49898 | 50000 | 127...
1,estimatedTotalAmt,string,0,12077,319386 | 292392 | 200000 | 49898 | 50000 | 127...
2,fundsObligated,string,1,15369,"['FY 2021 = $319,386.00'] | ['FY 2021 = $292,3..."
3,arraAmount,string,19912,4,740615 | 750000 | 749945 | 422913


In [13]:
numeric_amount_columns = [
    "fundsObligatedAmt",
    "estimatedTotalAmt",
    "arraAmount"
]

for column in numeric_amount_columns:
    nsf_clean_df[column] = pd.to_numeric(
        nsf_clean_df[column]
        .astype("string")
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.strip(),
        errors="coerce"
    ).astype("Float64")

for column in numeric_amount_columns:
    print(
        f"{column}: "
        f"{nsf_clean_df[column].isna().sum():,} missing/invalid | "
        f"dtype = {nsf_clean_df[column].dtype}"
    )

print(
    "\nNegative obligated amounts:",
    nsf_clean_df["fundsObligatedAmt"].lt(0).sum()
)

print(
    "Negative estimated totals:",
    nsf_clean_df["estimatedTotalAmt"].lt(0).sum()
)

display(
    nsf_clean_df[
        [
            "award_id_clean",
            "fundsObligatedAmt",
            "estimatedTotalAmt",
            "arraAmount",
            "currency"
        ]
    ].head()
)

fundsObligatedAmt: 0 missing/invalid | dtype = Float64
estimatedTotalAmt: 0 missing/invalid | dtype = Float64
arraAmount: 19,912 missing/invalid | dtype = Float64

Negative obligated amounts: 0
Negative estimated totals: 0


,award_id_clean,fundsObligatedAmt,estimatedTotalAmt,arraAmount,currency
0,2035038,319386.0,319386.0,<NA>,USD
1,2027234,292392.0,292392.0,<NA>,USD
2,2108526,200000.0,200000.0,<NA>,USD
3,2043611,49898.0,49898.0,<NA>,USD
4,2054691,50000.0,50000.0,<NA>,USD


In [14]:
amount_comparison_df = nsf_clean_df[
    [
        "award_id_clean",
        "fundsObligatedAmt",
        "estimatedTotalAmt"
    ]
].copy()

amount_comparison_df["amount_difference"] = (
    amount_comparison_df["estimatedTotalAmt"]
    - amount_comparison_df["fundsObligatedAmt"]
)

print(
    "Awards with equal obligated and estimated amounts:",
    f"{amount_comparison_df['amount_difference'].eq(0).sum():,}"
)

print(
    "Awards where estimated total is greater:",
    f"{amount_comparison_df['amount_difference'].gt(0).sum():,}"
)

print(
    "Awards where obligated amount is greater:",
    f"{amount_comparison_df['amount_difference'].lt(0).sum():,}"
)

print(
    "Largest positive difference:",
    f"${amount_comparison_df['amount_difference'].max():,.2f}"
)

print(
    "Largest negative difference:",
    f"${amount_comparison_df['amount_difference'].min():,.2f}"
)

display(
    amount_comparison_df.loc[
        amount_comparison_df["amount_difference"].ne(0)
    ]
    .sort_values(
        "amount_difference",
        ascending=False
    )
    .head(20)
)

Awards with equal obligated and estimated amounts: 15,657
Awards where estimated total is greater: 2,457
Awards where obligated amount is greater: 1,802
Largest positive difference: $167,474,836.00
Largest negative difference: $-17,023,408.00


,award_id_clean,fundsObligatedAmt,estimatedTotalAmt,amount_difference
7597,2435260,55190332.0,222665168.0,167474836.0
18338,2217817,268297104.0,415898912.0,147601808.0
9761,2323116,376000000.0,457447968.0,81447968.0
3809,2153337,15161000.0,91833000.0,76672000.0
10596,2413244,18113644.0,75000000.0,56886356.0
8836,2128556,156947248.0,195500000.0,38552752.0
19422,2420015,29976122.0,62096780.0,32120658.0
6178,2342336,31000000.0,55000000.0,24000000.0
3447,2153503,66886064.0,90800000.0,23913936.0
19426,2420044,22317120.0,45851224.0,23534104.0


In [15]:
amount_review_columns = [
    "award_id_clean",
    "title_clean",
    "activeAwd",
    "startDate",
    "expDate",
    "latestAmendmentDate",
    "fundsObligatedAmt",
    "estimatedTotalAmt"
]

largest_positive_differences_df = (
    nsf_clean_df[
        amount_review_columns
    ]
    .assign(
        amount_difference=(
            nsf_clean_df["estimatedTotalAmt"]
            - nsf_clean_df["fundsObligatedAmt"]
        )
    )
    .sort_values(
        "amount_difference",
        ascending=False
    )
    .head(10)
)

largest_negative_differences_df = (
    nsf_clean_df[
        amount_review_columns
    ]
    .assign(
        amount_difference=(
            nsf_clean_df["estimatedTotalAmt"]
            - nsf_clean_df["fundsObligatedAmt"]
        )
    )
    .sort_values(
        "amount_difference",
        ascending=True
    )
    .head(10)
)

print("Largest estimated-total amounts above obligated funding:")
display(largest_positive_differences_df)

print("\nLargest obligated amounts above estimated totals:")
display(largest_negative_differences_df)

Largest estimated-total amounts above obligated funding:


,award_id_clean,title_clean,activeAwd,startDate,expDate,latestAmendmentDate,fundsObligatedAmt,estimatedTotalAmt,amount_difference
7597,2435260,Research Infrastructure: National Geophysical ...,true,2025-10-01,2030-09-30,2026-07-07,55190332.0,222665168.0,167474836.0
18338,2217817,NEON Operations and Maintenance: Evolving from...,true,2023-11-01,2028-10-31,2026-07-27,268297104.0,415898912.0,147601808.0
9761,2323116,Construction for the Leadership Class Computin...,true,2024-07-01,2028-04-30,2026-07-23,376000000.0,457447968.0,81447968.0
3809,2153337,Mid-scale RI-2: Airborne Phased Array Radar (A...,false,2023-06-01,2025-07-31,2025-09-12,15161000.0,91833000.0,76672000.0
10596,2413244,Research Infrastructure: NSF Mid-scale RI-2: O...,true,2025-08-15,2030-07-31,2026-02-24,18113644.0,75000000.0,56886356.0
8836,2128556,National High Magnetic Field Laboratory Renewa...,true,2023-01-01,2027-12-31,2026-07-09,156947248.0,195500000.0,38552752.0
19422,2420015,R/V Atlantis 5 Year Proposal 2024-2028,true,2025-01-01,2028-12-31,2026-07-09,29976122.0,62096780.0,32120658.0
6178,2342336,Mid-scale: Operations of the Center for High E...,true,2024-09-01,2029-03-31,2026-06-12,31000000.0,55000000.0,24000000.0
3447,2153503,Mid-Scale RI-2 Consortium: Compact X-ray Free-...,true,2023-03-15,2028-02-29,2026-05-05,66886064.0,90800000.0,23913936.0
19426,2420044,R/V Neil Armstrong 5 Year Proposal 2024-2028,true,2025-01-01,2028-12-31,2026-07-16,22317120.0,45851224.0,23534104.0



Largest obligated amounts above estimated totals:


,award_id_clean,title_clean,activeAwd,startDate,expDate,latestAmendmentDate,fundsObligatedAmt,estimatedTotalAmt,amount_difference
10676,2505560,Category I: CloudBank 2: Accelerating Science ...,true,2025-08-01,2030-07-31,2026-07-05,37023408.0,20000000.0,-17023408.0
18468,2315305,NSF Engines: Textile Innovation Engine of Nort...,true,2024-03-01,2029-02-28,2026-07-09,31976680.0,15000000.0,-16976680.0
9546,2315760,NSF Engines: Colorado - Wyoming Climate Resili...,true,2024-03-01,2029-02-28,2026-07-29,30513596.0,15000000.0,-15513596.0
5185,2315320,NSF Engines: Central Florida Semiconductor Inn...,true,2024-03-01,2029-02-28,2026-06-24,29938672.0,14938671.0,-15000001.0
18480,2315695,NSF Engines: Upstate New York Energy Storage E...,true,2024-03-01,2029-02-28,2026-05-29,29869464.0,15000000.0,-14869464.0
5186,2315315,NSF Engines: North Dakota Advanced Agriculture...,true,2024-03-01,2029-02-28,2026-05-05,29494884.0,15000000.0,-14494884.0
18266,2320948,I-Corps: Continuation and Expansion of Trainin...,true,2023-10-01,2027-09-30,2026-06-29,24782404.0,12311663.0,-12470741.0
10685,2505376,Category I: AMA27: Sustainable Cyber-infrastru...,true,2025-08-01,2027-01-31,2026-05-07,21951104.0,11500508.0,-10450596.0
10563,2505662,Category I: Nexus: A Confluence of High-Perfor...,true,2025-07-01,2028-06-30,2026-07-26,28999998.0,20000000.0,-8999998.0
1355,2112356,Category II: ACES - Accelerating Computing for...,true,2021-10-01,2028-09-30,2026-04-21,12374786.0,5000000.0,-7374786.0


## Selecting the primary NSF award amount

The NSF dataset provides two main funding fields:

- `fundsObligatedAmt`, representing the amount formally obligated to date;
- `estimatedTotalAmt`, representing the estimated total value of the award.

The comparison showed that:

- 15,657 awards have the same value in both fields;
- 4,259 awards have different obligated and estimated totals;
- many of the largest differences occur for active, multi-year awards that have not yet received their full expected funding.

For GrantScopeAI, `estimatedTotalAmt` is selected as the primary award value because it better represents the expected total scale of each funded project.

The obligated amount is retained separately so that later analysis can distinguish between total expected funding and funding committed to date. The original funding fields are also preserved for traceability.

In [16]:
# Standardise NSF funding fields for the common grant schema

nsf_clean_df["amount_native"] = (
    nsf_clean_df["estimatedTotalAmt"]
)

nsf_clean_df["amount_obligated"] = (
    nsf_clean_df["fundsObligatedAmt"]
)

nsf_clean_df["amount_difference"] = (
    nsf_clean_df["amount_native"]
    - nsf_clean_df["amount_obligated"]
)

nsf_clean_df["amounts_differ"] = (
    nsf_clean_df["amount_difference"].ne(0)
)

nsf_clean_df["amount_source_field"] = (
    "estimatedTotalAmt"
)

print(
    "Missing primary award amounts:",
    nsf_clean_df["amount_native"].isna().sum()
)

print(
    "Awards with different estimated and obligated amounts:",
    f"{nsf_clean_df['amounts_differ'].sum():,}"
)

print(
    "Awards using estimatedTotalAmt as amount_native:",
    f"{nsf_clean_df['amount_source_field'].eq('estimatedTotalAmt').sum():,}"
)

display(
    nsf_clean_df[
        [
            "award_id_clean",
            "amount_native",
            "amount_obligated",
            "amount_difference",
            "currency"
        ]
    ].head()
)

Missing primary award amounts: 0
Awards with different estimated and obligated amounts: 4,259
Awards using estimatedTotalAmt as amount_native: 19,916


,award_id_clean,amount_native,amount_obligated,amount_difference,currency
0,2035038,319386.0,319386.0,0.0,USD
1,2027234,292392.0,292392.0,0.0,USD
2,2108526,200000.0,200000.0,0.0,USD
3,2043611,49898.0,49898.0,0.0,USD
4,2054691,50000.0,50000.0,0.0,USD


In [17]:
nsf_program_columns = [
    "fundProgramName",
    "primaryProgram",
    "program",
    "dirAbbr",
    "divAbbr",
    "orgLongName",
    "orgLongName2"
]

program_field_summary = []

for column in nsf_program_columns:
    non_missing_values = (
        nsf_clean_df[column]
        .dropna()
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    program_field_summary.append({
        "column": column,
        "missing_rows": nsf_clean_df[column].isna().sum(),
        "missing_percent": round(
            nsf_clean_df[column].isna().mean() * 100,
            2
        ),
        "unique_values": nsf_clean_df[column].nunique(dropna=True),
        "examples": " | ".join(
            non_missing_values.head(5).tolist()
        )
    })

program_field_summary_df = pd.DataFrame(
    program_field_summary
)

display(program_field_summary_df)

,column,missing_rows,missing_percent,unique_values,examples
0,fundProgramName,32,0.16,3215,AM-Advanced Manufacturing | Capacity: Cyberinf...
1,primaryProgram,0,0.00,1687,['01002122DB NSF RESEARCH & RELATED ACTIVIT'] ...
2,program,1903,9.56,6948,"Advanced Manufacturing, TOOLS & TECHNOL FOR MA..."
3,dirAbbr,0,0.00,9,ENG | BIO | BIO | CSE | TIP
4,divAbbr,0,0.00,40,CMMI | DBI | DEB | CNS | TI
5,orgLongName,0,0.00,9,Directorate for Engineering | Directorate for ...
6,orgLongName2,0,0.00,40,"Division of Civil, Mechanical, and Manufacturi..."


In [18]:
program_format_check_df = pd.DataFrame({
    "check": [
        "primaryProgram values beginning with [",
        "primaryProgram values containing multiple list items",
        "program values containing commas",
        "program values containing semicolons",
        "fundProgramName missing values"
    ],
    "row_count": [
        nsf_clean_df["primaryProgram"]
        .str.startswith("[", na=False)
        .sum(),

        nsf_clean_df["primaryProgram"]
        .str.contains(r"',\s*'", na=False, regex=True)
        .sum(),

        nsf_clean_df["program"]
        .str.contains(",", na=False, regex=False)
        .sum(),

        nsf_clean_df["program"]
        .str.contains(";", na=False, regex=False)
        .sum(),

        nsf_clean_df["fundProgramName"]
        .isna()
        .sum()
    ]
})

display(program_format_check_df)

print("\nExample primaryProgram values:")
display(
    nsf_clean_df[
        ["award_id_clean", "primaryProgram"]
    ]
    .drop_duplicates("primaryProgram")
    .head(15)
)

print("\nExample program values:")
display(
    nsf_clean_df[
        ["award_id_clean", "program"]
    ]
    .dropna()
    .drop_duplicates("program")
    .head(15)
)

,check,row_count
0,primaryProgram values beginning with [,19916
1,primaryProgram values containing multiple list...,6127
2,program values containing commas,13080
3,program values containing semicolons,0
4,fundProgramName missing values,32



Example primaryProgram values:


,award_id_clean,primaryProgram
0,2035038,['01002122DB NSF RESEARCH & RELATED ACTIVIT']
2,2108526,['01002021RB NSF RESEARCH & RELATED ACTIVIT']
6,2045829,"['01002122DB NSF RESEARCH & RELATED ACTIVIT', ..."
9,2124722,"['01002021DB NSF RESEARCH & RELATED ACTIVIT', ..."
11,2110145,['01001920DB NSF RESEARCH & RELATED ACTIVIT']
16,2138834,"['01002324DB NSF RESEARCH & RELATED ACTIVIT', ..."
18,2038853,"['01002223DB NSF RESEARCH & RELATED ACTIVIT', ..."
19,2141037,"['01002122DB NSF RESEARCH & RELATED ACTIVIT', ..."
20,2126602,"['01002324DB NSF RESEARCH & RELATED ACTIVIT', ..."
21,2034385,['01002021DB NSF RESEARCH & RELATED ACTIVIT']



Example program values:


,award_id_clean,program
0,2035038,"Advanced Manufacturing, TOOLS & TECHNOL FOR MA..."
1,2027234,"ADVANCES IN BIO INFORMATICS, EXP PROG TO STIM ..."
2,2108526,"COVID-19 Research, RAPID"
3,2043611,S&CC: Smart and Connected Communities
4,2054691,"Advanced Manufacturing, ARTIFICIAL INTELL & CO..."
5,2041221,"LINGUISTICS, GRADUATE INVOLVEMENT, SCIENCE, MA..."
6,2045829,"Control systems & applications, CAREER-Faculty..."
7,1951852,"S&CC: Smart and Connected Communities, REU SUP..."
9,2124722,CAREER-Faculty Erly Career Dev
10,2003444,"Materials AI, BIO-RELATED MATERIALS RESEARCH"


In [19]:
missing_fund_program_df = (
    nsf_clean_df.loc[
        nsf_clean_df["fundProgramName"].isna(),
        [
            "award_id_clean",
            "title_clean",
            "fundProgramName",
            "program",
            "primaryProgram",
            "dirAbbr",
            "divAbbr",
            "orgLongName",
            "orgLongName2"
        ]
    ]
    .copy()
)

print(
    "Awards missing fundProgramName:",
    f"{len(missing_fund_program_df):,}"
)

print(
    "Missing fundProgramName but available program:",
    f"{missing_fund_program_df['program'].notna().sum():,}"
)

print(
    "Missing both fundProgramName and program:",
    f"{missing_fund_program_df['program'].isna().sum():,}"
)

display(missing_fund_program_df.head(20))

Awards missing fundProgramName: 32
Missing fundProgramName but available program: 26
Missing both fundProgramName and program: 6


,award_id_clean,title_clean,fundProgramName,program,primaryProgram,dirAbbr,divAbbr,orgLongName,orgLongName2
2,2108526,RAPID: Real-time Forecasting of COVID-19 risk ...,<NA>,"COVID-19 Research, RAPID",['01002021RB NSF RESEARCH & RELATED ACTIVIT'],BIO,DEB,Directorate for Biological Sciences,Division Of Environmental Biology
494,2134890,Bureau of Transportation Statistics (BTS) Supp...,<NA>,NAT CENTER FOR ATMOSPHERIC RES,['01002122RB NSF RESEARCH & RELATED ACTIVIT'],GEO,AGS,Directorate for Geosciences,Division of Atmospheric and Geospace Sciences
756,2124535,ATD: Inductive Spatiotemporal Graph Encoding f...,<NA>,ALGORITHMS IN THREAT DETECTION,['01002122RB NSF RESEARCH & RELATED ACTIVIT'],MPS,DMS,Directorate for Mathematical and Physical Scie...,Division Of Mathematical Sciences
847,2135784,RAPID: Fast COVID-19 Scenario Projections in P...,<NA>,"COVID-19 Research, RAPID",['01002021RB NSF RESEARCH & RELATED ACTIVIT'],BIO,DEB,Directorate for Biological Sciences,Division Of Environmental Biology
1022,2027737,ATD: Algorithmic Threat Detection and Mitigati...,<NA>,"Artificial Intelligence (AI), Machine Learning...",['01002122RB NSF RESEARCH & RELATED ACTIVIT'],MPS,DMS,Directorate for Mathematical and Physical Scie...,Division Of Mathematical Sciences
2075,2223933,RAPID: Data-driven Understanding of Imperfect ...,<NA>,"COVID-19 Research, RAPID",['01002122RB NSF RESEARCH & RELATED ACTIVIT'],BIO,DEB,Directorate for Biological Sciences,Division Of Environmental Biology
2294,2226232,NSF Convergence Accelerator Track G: Proactive...,<NA>,<NA>,['01002122RB NSF RESEARCH & RELATED ACTIVIT'],TIP,ITE,"Directorate for Technology, Innovation, and Pa...",Innovation and Technology Ecosystems
2298,2226443,NSF Convergence Accelerator Track: G: Security...,<NA>,<NA>,['01002122RB NSF RESEARCH & RELATED ACTIVIT'],TIP,ITE,"Directorate for Technology, Innovation, and Pa...",Innovation and Technology Ecosystems
2619,2226437,NSF Convergence Accelerator Track G: SONIC: Se...,<NA>,<NA>,['01002122RB NSF RESEARCH & RELATED ACTIVIT'],TIP,ITE,"Directorate for Technology, Innovation, and Pa...",Innovation and Technology Ecosystems
2733,2200169,PIPP Phase I: Center for Pandemic Decision Sci...,<NA>,"COVID-19 Research, PIPP-Pandemic Prevention",['01002122RB NSF RESEARCH & RELATED ACTIVIT'],BIO,DEB,Directorate for Biological Sciences,Division Of Environmental Biology


In [20]:
missing_both_program_fields_df = (
    missing_fund_program_df.loc[
        missing_fund_program_df["program"].isna(),
        [
            "award_id_clean",
            "title_clean",
            "primaryProgram",
            "dirAbbr",
            "divAbbr",
            "orgLongName",
            "orgLongName2"
        ]
    ]
    .copy()
)

display(missing_both_program_fields_df)

,award_id_clean,title_clean,primaryProgram,dirAbbr,divAbbr,orgLongName,orgLongName2
2294,2226232,NSF Convergence Accelerator Track G: Proactive...,['01002122RB NSF RESEARCH & RELATED ACTIVIT'],TIP,ITE,"Directorate for Technology, Innovation, and Pa...",Innovation and Technology Ecosystems
2298,2226443,NSF Convergence Accelerator Track: G: Security...,['01002122RB NSF RESEARCH & RELATED ACTIVIT'],TIP,ITE,"Directorate for Technology, Innovation, and Pa...",Innovation and Technology Ecosystems
2619,2226437,NSF Convergence Accelerator Track G: SONIC: Se...,['01002122RB NSF RESEARCH & RELATED ACTIVIT'],TIP,ITE,"Directorate for Technology, Innovation, and Pa...",Innovation and Technology Ecosystems
4139,2333494,RAPID: Retrospective COVID-19 Scenario Project...,['01002223RB NSF RESEARCH & RELATED ACTIVIT'],BIO,DEB,Directorate for Biological Sciences,Division Of Environmental Biology
8563,2226423,NSF Convergence Accelerator Track G: Secure Te...,['01002122RB NSF RESEARCH & RELATED ACTIVIT'],TIP,ITE,"Directorate for Technology, Innovation, and Pa...",Innovation and Technology Ecosystems
17421,2226457,NSF Convergence Accelerator Track G: Privacy-p...,['01002122RB NSF RESEARCH & RELATED ACTIVIT'],TIP,ITE,"Directorate for Technology, Innovation, and Pa...",Innovation and Technology Ecosystems


In [21]:
# Standardise NSF programme and organisational-unit fields

nsf_clean_df["programme_name"] = (
    nsf_clean_df["fundProgramName"]
    .fillna(nsf_clean_df["program"])
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

nsf_clean_df["programme_name_source"] = "fundProgramName"

nsf_clean_df.loc[
    nsf_clean_df["fundProgramName"].isna()
    & nsf_clean_df["program"].notna(),
    "programme_name_source"
] = "program_fallback"

nsf_clean_df.loc[
    nsf_clean_df["programme_name"].isna(),
    "programme_name_source"
] = "missing"


nsf_clean_df["directorate_name"] = (
    nsf_clean_df["orgLongName"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

nsf_clean_df["division_name"] = (
    nsf_clean_df["orgLongName2"]
    .astype("string")
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

nsf_clean_df["directorate_code"] = (
    nsf_clean_df["dirAbbr"]
    .astype("string")
    .str.strip()
)

nsf_clean_df["division_code"] = (
    nsf_clean_df["divAbbr"]
    .astype("string")
    .str.strip()
)


print(
    "Programme names from fundProgramName:",
    f"{nsf_clean_df['programme_name_source'].eq('fundProgramName').sum():,}"
)

print(
    "Programme names using program fallback:",
    f"{nsf_clean_df['programme_name_source'].eq('program_fallback').sum():,}"
)

print(
    "Programme names still missing:",
    f"{nsf_clean_df['programme_name'].isna().sum():,}"
)

print(
    "Missing directorate names:",
    nsf_clean_df["directorate_name"].isna().sum()
)

print(
    "Missing division names:",
    nsf_clean_df["division_name"].isna().sum()
)

display(
    nsf_clean_df[
        [
            "award_id_clean",
            "programme_name",
            "programme_name_source",
            "directorate_code",
            "directorate_name",
            "division_code",
            "division_name"
        ]
    ].head()
)

Programme names from fundProgramName: 19,884
Programme names using program fallback: 26
Programme names still missing: 6
Missing directorate names: 0
Missing division names: 0


,award_id_clean,programme_name,programme_name_source,directorate_code,directorate_name,division_code,division_name
0,2035038,AM-Advanced Manufacturing,fundProgramName,ENG,Directorate for Engineering,CMMI,"Division of Civil, Mechanical, and Manufacturi..."
1,2027234,Capacity: Cyberinfrastructure,fundProgramName,BIO,Directorate for Biological Sciences,DBI,Division of Biological Infrastructure
2,2108526,"COVID-19 Research, RAPID",program_fallback,BIO,Directorate for Biological Sciences,DEB,Division Of Environmental Biology
3,2043611,S&CC: Smart & Connected Commun,fundProgramName,CSE,Directorate for Computer and Information Scien...,CNS,Division Of Computer and Network Systems
4,2054691,I-Corps,fundProgramName,TIP,"Directorate for Technology, Innovation, and Pa...",TI,Translational Impacts


In [22]:
nsf_location_columns = [
    "awardeeName",
    "awardee",
    "awardeeCity",
    "awardeeStateCode",
    "awardeeCountryCode",
    "perfCity",
    "perfStateCode",
    "perfCountryCode",
    "ueiNumber",
    "parentUeiNumber"
]

location_field_summary = []

for column in nsf_location_columns:
    non_missing_values = (
        nsf_clean_df[column]
        .dropna()
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    location_field_summary.append({
        "column": column,
        "missing_rows": nsf_clean_df[column].isna().sum(),
        "missing_percent": round(
            nsf_clean_df[column].isna().mean() * 100,
            2
        ),
        "unique_values": nsf_clean_df[column].nunique(dropna=True),
        "examples": " | ".join(
            non_missing_values.head(5).tolist()
        )
    })

location_field_summary_df = pd.DataFrame(
    location_field_summary
)

display(location_field_summary_df)

,column,missing_rows,missing_percent,unique_values,examples
0,awardeeName,0,0.00,2096,Virginia Polytechnic Institute and State Unive...
1,awardee,240,1.21,1784,VIRGINIA POLYTECHNIC INSTITUTE & STATE UNIVERS...
2,awardeeCity,0,0.00,970,BLACKSBURG | GAINESVILLE | BALTIMORE | DETROIT...
3,awardeeStateCode,22,0.11,54,VA | FL | MD | MI | MI
4,awardeeCountryCode,0,0.00,8,US | US | US | US | US
5,perfCity,952,4.78,1404,Blacksburg | Gainesville | Baltimore | Detroit...
6,perfStateCode,39,0.20,55,VA | FL | MD | MI | MI
7,perfCountryCode,1,0.01,19,US | US | US | US | US
8,ueiNumber,236,1.18,1861,QDE5UHE5XD16 | NNFQH1JAPEP3 | FTMTDMBR29C7 | M...
9,parentUeiNumber,11581,58.15,446,X6KEFGLHSJX7 | GS4PNKTRNKL3 | GS4PNKTRNKL3 | Q...


In [23]:
# Standardise recipient organisation and research-performance locations

def clean_text_field(series):
    return (
        series
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )


# Primary recipient organisation
nsf_clean_df["organisation_name"] = clean_text_field(
    nsf_clean_df["awardeeName"]
)

nsf_clean_df["organisation_city"] = clean_text_field(
    nsf_clean_df["awardeeCity"]
)

nsf_clean_df["organisation_state_code"] = clean_text_field(
    nsf_clean_df["awardeeStateCode"]
)

nsf_clean_df["organisation_country_code"] = clean_text_field(
    nsf_clean_df["awardeeCountryCode"]
)

nsf_clean_df["organisation_uei"] = clean_text_field(
    nsf_clean_df["ueiNumber"]
)

nsf_clean_df["parent_organisation_uei"] = clean_text_field(
    nsf_clean_df["parentUeiNumber"]
)


# Location where the funded work is performed
nsf_clean_df["performance_city"] = clean_text_field(
    nsf_clean_df["perfCity"]
)

nsf_clean_df["performance_state_code"] = clean_text_field(
    nsf_clean_df["perfStateCode"]
)

nsf_clean_df["performance_country_code"] = clean_text_field(
    nsf_clean_df["perfCountryCode"]
)


location_validation_df = pd.DataFrame({
    "field": [
        "organisation_name",
        "organisation_city",
        "organisation_state_code",
        "organisation_country_code",
        "organisation_uei",
        "performance_city",
        "performance_state_code",
        "performance_country_code"
    ],
    "missing_rows": [
        nsf_clean_df["organisation_name"].isna().sum(),
        nsf_clean_df["organisation_city"].isna().sum(),
        nsf_clean_df["organisation_state_code"].isna().sum(),
        nsf_clean_df["organisation_country_code"].isna().sum(),
        nsf_clean_df["organisation_uei"].isna().sum(),
        nsf_clean_df["performance_city"].isna().sum(),
        nsf_clean_df["performance_state_code"].isna().sum(),
        nsf_clean_df["performance_country_code"].isna().sum()
    ]
})

display(location_validation_df)

display(
    nsf_clean_df[
        [
            "award_id_clean",
            "organisation_name",
            "organisation_city",
            "organisation_state_code",
            "organisation_country_code",
            "organisation_uei",
            "performance_city",
            "performance_state_code",
            "performance_country_code"
        ]
    ].head()
)

,field,missing_rows
0,organisation_name,0
1,organisation_city,0
2,organisation_state_code,22
3,organisation_country_code,0
4,organisation_uei,236
5,performance_city,952
6,performance_state_code,39
7,performance_country_code,1


,award_id_clean,organisation_name,organisation_city,organisation_state_code,organisation_country_code,organisation_uei,performance_city,performance_state_code,performance_country_code
0,2035038,Virginia Polytechnic Institute and State Unive...,BLACKSBURG,VA,US,QDE5UHE5XD16,Blacksburg,VA,US
1,2027234,University of Florida,GAINESVILLE,FL,US,NNFQH1JAPEP3,Gainesville,FL,US
2,2108526,Johns Hopkins University,BALTIMORE,MD,US,FTMTDMBR29C7,Baltimore,MD,US
3,2043611,Wayne State University,DETROIT,MI,US,M6K6NTJ2MNE5,Detroit,MI,US
4,2054691,Wayne State University,DETROIT,MI,US,M6K6NTJ2MNE5,Detroit,MI,US


In [24]:
# Create official NSF award URLs and standardise award status

nsf_clean_df["source_url"] = (
    "https://www.nsf.gov/awardsearch/showAward"
    "?AWD_ID="
    + nsf_clean_df["award_id_clean"]
)

nsf_clean_df["is_active_award"] = (
    nsf_clean_df["activeAwd"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map({
        "true": True,
        "false": False
    })
    .astype("boolean")
)

print(
    "Missing source URLs:",
    nsf_clean_df["source_url"].isna().sum()
)

print(
    "Missing or invalid active-award values:",
    nsf_clean_df["is_active_award"].isna().sum()
)

print(
    "Active awards:",
    f"{nsf_clean_df['is_active_award'].eq(True).sum():,}"
)

print(
    "Inactive awards:",
    f"{nsf_clean_df['is_active_award'].eq(False).sum():,}"
)

display(
    nsf_clean_df[
        [
            "award_id_clean",
            "title_clean",
            "is_active_award",
            "source_url"
        ]
    ].head()
)

Missing source URLs: 0
Missing or invalid active-award values: 0
Active awards: 10,637
Inactive awards: 9,279


,award_id_clean,title_clean,is_active_award,source_url
0,2035038,Ultra-high Precision Assembly of Aerospace Com...,False,https://www.nsf.gov/awardsearch/showAward?AWD_...
1,2027234,Collaborative Research: CIBR: Leaping the Spec...,False,https://www.nsf.gov/awardsearch/showAward?AWD_...
2,2108526,RAPID: Real-time Forecasting of COVID-19 risk ...,False,https://www.nsf.gov/awardsearch/showAward?AWD_...
3,2043611,SCC-CIVIC-PG Track A: Leveraging AI-assist Mic...,False,https://www.nsf.gov/awardsearch/showAward?AWD_...
4,2054691,I-Corps: AI-enabled automation intelligence so...,False,https://www.nsf.gov/awardsearch/showAward?AWD_...


In [25]:
nsf_pi_columns = [
    "pdPIName",
    "piFirstName",
    "piMiddeInitial",
    "piLastName",
    "piEmail",
    "piId",
    "coPDPI"
]

pi_field_summary = []

for column in nsf_pi_columns:
    non_missing_values = (
        nsf_clean_df[column]
        .dropna()
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    pi_field_summary.append({
        "column": column,
        "missing_rows": nsf_clean_df[column].isna().sum(),
        "missing_percent": round(
            nsf_clean_df[column].isna().mean() * 100,
            2
        ),
        "unique_values": nsf_clean_df[column].nunique(dropna=True),
        "examples": " | ".join(
            non_missing_values.head(5).tolist()
        )
    })

pi_field_summary_df = pd.DataFrame(pi_field_summary)

display(pi_field_summary_df)

,column,missing_rows,missing_percent,unique_values,examples
0,pdPIName,1,0.01,15093,Zhenyu Kong | Robert P Guralnick | Lauren M Ga...
1,piFirstName,1,0.01,6107,Zhenyu | Robert | Lauren | Dongxiao | Jeremy
2,piMiddeInitial,12228,61.40,33,P | M | R | A | A
3,piLastName,1,0.01,9597,Kong | Guralnick | Gardner | Zhu | Rickli
4,piEmail,240,1.21,15018,zkong@vt.edu | robgur@gmail.com | l.gardner@jh...
5,piId,5,0.03,15275,269797976 | 269942561 | 270025125 | 269836722 ...
6,coPDPI,12741,63.97,6976,"['Daniel Grosu dgrosu@wayne.edu', 'Tierra Bill..."


In [26]:
# Standardise principal-investigator information

constructed_pi_name = (
    nsf_clean_df["piFirstName"]
    .fillna("")
    .astype("string")
    .str.strip()
    + " "
    + nsf_clean_df["piLastName"]
    .fillna("")
    .astype("string")
    .str.strip()
)

constructed_pi_name = (
    constructed_pi_name
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)

nsf_clean_df["lead_investigator"] = (
    clean_text_field(nsf_clean_df["pdPIName"])
    .fillna(constructed_pi_name)
)

nsf_clean_df["lead_investigator_id"] = (
    clean_text_field(nsf_clean_df["piId"])
)

nsf_clean_df["lead_investigator_email"] = (
    clean_text_field(nsf_clean_df["piEmail"])
)

# Preserve the source co-investigator field as lineage text
nsf_clean_df["co_investigators_raw"] = (
    clean_text_field(nsf_clean_df["coPDPI"])
)

print(
    "Missing lead-investigator names:",
    nsf_clean_df["lead_investigator"].isna().sum()
)

print(
    "Missing lead-investigator IDs:",
    nsf_clean_df["lead_investigator_id"].isna().sum()
)

print(
    "Missing lead-investigator emails:",
    nsf_clean_df["lead_investigator_email"].isna().sum()
)

print(
    "Awards listing co-investigators:",
    f"{nsf_clean_df['co_investigators_raw'].notna().sum():,}"
)

display(
    nsf_clean_df[
        [
            "award_id_clean",
            "lead_investigator",
            "lead_investigator_id",
            "lead_investigator_email",
            "co_investigators_raw"
        ]
    ].head()
)

Missing lead-investigator names: 1
Missing lead-investigator IDs: 5
Missing lead-investigator emails: 240
Awards listing co-investigators: 7,175


,award_id_clean,lead_investigator,lead_investigator_id,lead_investigator_email,co_investigators_raw
0,2035038,Zhenyu Kong,269797976,zkong@vt.edu,<NA>
1,2027234,Robert P Guralnick,269942561,robgur@gmail.com,<NA>
2,2108526,Lauren M Gardner,270025125,l.gardner@jhu.edu,<NA>
3,2043611,Dongxiao Zhu,269836722,dzhu@wayne.edu,"['Daniel Grosu dgrosu@wayne.edu', 'Tierra Bill..."
4,2054691,Jeremy Rickli,269943221,jlrickli@wayne.edu,['Murat Yildirim murat@wayne.edu']


## Defining the text used for relevance classification

GrantScopeAI needs to identify NSF awards that combine artificial intelligence with chemistry or materials research.

The relevance search text is built from:

- the cleaned award title;
- the cleaned award abstract;
- the NSF programme name;
- the broader programme field;
- the directorate and division names.

The extraction-query fields are deliberately excluded from the classification text. Because the raw awards were originally collected using topic-based queries, including those query labels would create circular logic and could cause awards to appear relevant simply because of how they were retrieved.

All awards are also checked against the project date range of 2021–2025 before relevance tiers are assigned.

In [27]:
# Confirm the project date scope
nsf_clean_df["in_project_date_scope"] = (
    nsf_clean_df["award_year"].between(2021, 2025)
)

# Combine award content and programme metadata for searching
nsf_search_columns = [
    "title_clean",
    "abstract_clean",
    "programme_name",
    "program",
    "directorate_name",
    "division_name"
]

nsf_clean_df["search_text"] = (
    nsf_clean_df[nsf_search_columns]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(
    "Awards in 2021–2025 scope:",
    f"{nsf_clean_df['in_project_date_scope'].sum():,}"
)

print(
    "Awards outside scope:",
    f"{(~nsf_clean_df['in_project_date_scope']).sum():,}"
)

print(
    "Empty search-text rows:",
    nsf_clean_df["search_text"].eq("").sum()
)

print(
    "Median search-text length:",
    int(nsf_clean_df["search_text"].str.len().median())
)

Awards in 2021–2025 scope: 19,916
Awards outside scope: 0
Empty search-text rows: 0
Median search-text length: 3171


## Creating the broad relevance filter

The first relevance filter is intentionally designed for high recall. Its purpose is to identify a broad pool of awards that mention both:

- artificial intelligence or data-driven methods; and
- chemistry, molecular science, or materials-related concepts.

An award is classified as a baseline match only when it contains at least one term from each group.

The initial keyword lists include broad terms such as `data-driven`, `predictive modeling`, `molecular`, `synthesis`, and `spectroscopy`. These terms help avoid missing potentially relevant projects, but they can also appear in unrelated fields.

The baseline results are therefore treated as a candidate pool rather than the final classification. A stricter filter and manual sample review are used in later steps to improve precision.

In [28]:
# Build relevance text from award content and programme information.
# matched_queries is excluded to avoid circular classification.

nsf_clean_df["relevance_text_core"] = (
    nsf_clean_df[
        [
            "title_clean",
            "abstract_clean",
            "programme_name",
            "program"
        ]
    ]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


nsf_ai_pattern = (
    r"\b(?:"
    r"artificial intelligence|"
    r"machine learning|"
    r"deep learning|"
    r"neural networks?|"
    r"generative ai|"
    r"scientific machine learning|"
    r"reinforcement learning|"
    r"computer vision|"
    r"natural language processing|"
    r"foundation models?|"
    r"data[- ]driven|"
    r"predictive modelling|"
    r"predictive modeling|"
    r"ai"
    r")\b"
)


nsf_chemistry_materials_pattern = (
    r"\b(?:"
    r"chemistry|chemical|chemicals|"
    r"molecular|molecules?|"
    r"catalysis|catalysts?|catalytic|"
    r"polymers?|"
    r"materials science|"
    r"materials discovery|"
    r"advanced materials|"
    r"reaction prediction|"
    r"chemical reactions?|"
    r"synthesis|"
    r"synthetic chemistry|"
    r"computational chemistry|"
    r"quantum chemistry|"
    r"electrochemistry|"
    r"electrochemical|"
    r"spectroscopy|"
    r"drug discovery|"
    r"battery materials?|"
    r"nanomaterials?|"
    r"biomaterials?"
    r")\b"
)


nsf_clean_df["ai_keyword_match"] = (
    nsf_clean_df["relevance_text_core"]
    .str.contains(
        nsf_ai_pattern,
        case=False,
        na=False,
        regex=True
    )
)

nsf_clean_df["chemistry_materials_keyword_match"] = (
    nsf_clean_df["relevance_text_core"]
    .str.contains(
        nsf_chemistry_materials_pattern,
        case=False,
        na=False,
        regex=True
    )
)

nsf_both_keyword_match = (
    nsf_clean_df["ai_keyword_match"]
    & nsf_clean_df["chemistry_materials_keyword_match"]
)

nsf_clean_df["baseline_relevant"] = (
    nsf_clean_df["in_project_date_scope"]
    & nsf_both_keyword_match
)


ai_match_count = nsf_clean_df["ai_keyword_match"].sum()

chemistry_match_count = (
    nsf_clean_df["chemistry_materials_keyword_match"].sum()
)

both_match_count = nsf_both_keyword_match.sum()

baseline_candidate_count = (
    nsf_clean_df["baseline_relevant"].sum()
)


print(
    "Awards matching AI terms:",
    f"{ai_match_count:,}"
)

print(
    "Awards matching chemistry/materials terms:",
    f"{chemistry_match_count:,}"
)

print(
    "Awards matching both term groups:",
    f"{both_match_count:,}"
)

print(
    "Baseline-relevant awards:",
    f"{baseline_candidate_count:,}"
)

Awards matching AI terms: 11,364
Awards matching chemistry/materials terms: 7,529
Awards matching both term groups: 2,624
Baseline-relevant awards: 2,624


In [29]:
import re


def extract_keyword_matches(text, pattern):
    if pd.isna(text):
        return ""

    matches = {
        match.group(0).lower()
        for match in re.finditer(
            pattern,
            str(text),
            flags=re.IGNORECASE
        )
    }

    return " | ".join(sorted(matches))


nsf_relevance_review_sample_df = (
    nsf_clean_df.loc[
        nsf_clean_df["baseline_relevant"]
    ]
    .sample(
        n=20,
        random_state=42
    )
    .copy()
)

nsf_relevance_review_sample_df["ai_matches"] = (
    nsf_relevance_review_sample_df["relevance_text_core"]
    .apply(
        lambda text: extract_keyword_matches(
            text,
            nsf_ai_pattern
        )
    )
)

nsf_relevance_review_sample_df[
    "chemistry_materials_matches"
] = (
    nsf_relevance_review_sample_df["relevance_text_core"]
    .apply(
        lambda text: extract_keyword_matches(
            text,
            nsf_chemistry_materials_pattern
        )
    )
)

nsf_relevance_review_sample_df["abstract_preview"] = (
    nsf_relevance_review_sample_df["abstract_clean"]
    .str.slice(0, 350)
)

display(
    nsf_relevance_review_sample_df[
        [
            "award_id_clean",
            "award_year",
            "title_clean",
            "programme_name",
            "ai_matches",
            "chemistry_materials_matches",
            "abstract_preview"
        ]
    ]
)

,award_id_clean,award_year,title_clean,programme_name,ai_matches,chemistry_materials_matches,abstract_preview
1372,2119172,2021,DMREF: Engineering the On-The-Fly Control of 3...,"Catalysis, DMREF, Proc Sys, Reac Eng & Mol Therm",ai | machine learning,catalysis | chemical | chemistry | materials d...,Biology is capable of creating materials with ...
9459,2344344,2024,NSF Convergence Accelerator Track L: An Integr...,Convergence Accelerator Resrch,artificial intelligence,molecular,This research addresses the pressing public he...
6152,2407565,2024,Collaborative Research: Studying Nearby Cosmic...,STELLAR ASTRONOMY & ASTROPHYSC,machine learning,chemical | spectroscopy,"Exploding stars, called supernovae, have an ou..."
2383,2141660,2022,CAREER: Integrative Pathway Analysis for Cance...,Info Integration & Informatics,machine learning,molecular,Cancer is an umbrella term that includes a ran...
12032,2347727,2024,ERI: SoundEYE: AI-Driven Sensory Augmentation ...,"EDSE-Engineering Design and Sy, ERI-Eng. Resea...",ai,synthesis,The goal of this Engineering Research Initiati...
3902,2306092,2023,An engineered platform to establish the role o...,Engineering of Biomed Systems,machine learning,molecules,"Despite improvements to the standard of care, ..."
13665,2118180,2021,Collaborative Research: CyberTraining: Impleme...,CyberTraining - Training-based,machine learning,chemical | chemistry | materials science | mol...,Computational research in molecular sciences i...
229,2047034,2021,Career: Correct-by-Learning Methods for Reliab...,"Software & Hardware Foundation, CPS-Cyber-Phys...",ai | data-driven | machine learning,synthesis,Computing systems that engage people physicall...
470,2132672,2021,I-Corps: Developing predictive computational m...,I-Corps,ai | artificial intelligence,chemistry | drug discovery | molecule,The broader impact/commercial potential of thi...
3417,2222074,2023,Organotypic whole hemisphere models to probe s...,Engineering of Biomed Systems,machine learning | neural network,molecular,The brain is our most complex organ and govern...


## Refining the relevance filter for higher precision

The broad filter captures potentially relevant awards, but manual review showed that some terms create false positives.

Examples include:

- `synthesis` referring to software, robotics, or control-system design;
- `molecular` appearing in biomedical projects without a central chemistry focus;
- `spectroscopy` appearing in astronomy or physics;
- `chemical` being mentioned only as a measured substance.

A stricter filter is therefore created using more specific AI and chemistry/materials terminology.

The refined filter requires:

- a strong AI method, such as machine learning, deep learning, neural networks, reinforcement learning, or artificial intelligence; and
- a clearly defined chemistry or materials concept, such as catalysis, polymers, materials design, computational chemistry, electrochemistry, molecular modelling, or drug discovery.

The uppercase `AI` acronym is matched separately using case-sensitive logic to reduce accidental matches with ordinary text.

Awards that pass this stricter rule are treated as higher-confidence candidates, while awards found only by the broad filter remain available for review.

In [30]:
# Strong AI terminology
nsf_strong_ai_pattern = (
    r"\b(?:"
    r"artificial intelligence|"
    r"machine learning|"
    r"deep learning|"
    r"neural networks?|"
    r"generative ai|"
    r"scientific machine learning|"
    r"reinforcement learning|"
    r"computer vision|"
    r"natural language processing|"
    r"foundation models?|"
    r"materials informatics|"
    r"cheminformatics"
    r")\b"
)

nsf_full_ai_match = (
    nsf_clean_df["relevance_text_core"]
    .str.contains(
        nsf_strong_ai_pattern,
        case=False,
        na=False,
        regex=True
    )
)

# Match the uppercase AI acronym separately
nsf_ai_acronym_match = (
    nsf_clean_df["relevance_text_core"]
    .str.contains(
        r"\bAI(?:[- ](?:driven|enabled|based|assisted|powered))?\b",
        case=True,
        na=False,
        regex=True
    )
)

nsf_clean_df["strong_ai_match"] = (
    nsf_full_ai_match
    | nsf_ai_acronym_match
)


# Higher-precision chemistry and materials terminology
nsf_strong_chemistry_materials_pattern = (
    r"\b(?:"
    r"chemistry|"
    r"chemical engineering|"
    r"chemical synthesis|"
    r"chemical reactions?|"
    r"chemical processes?|"
    r"catalysis|catalysts?|catalytic|"
    r"polymers?|polymeric|"
    r"materials science|"
    r"materials discovery|"
    r"materials design|"
    r"materials modell?ing|"
    r"materials simulation|"
    r"materials informatics|"
    r"advanced materials|"
    r"reaction prediction|"
    r"organic synthesis|"
    r"computational chemistry|"
    r"quantum chemistry|"
    r"cheminformatics|"
    r"molecular modell?ing|"
    r"molecular design|"
    r"molecular simulation|"
    r"molecular dynamics|"
    r"molecular property prediction|"
    r"electrochemistry|electrochemical|"
    r"drug discovery|"
    r"battery materials?|"
    r"nanomaterials?|"
    r"biomaterials?|"
    r"supramolecular|"
    r"macromolecular"
    r")\b"
)

nsf_clean_df["strong_chemistry_materials_match"] = (
    nsf_clean_df["relevance_text_core"]
    .str.contains(
        nsf_strong_chemistry_materials_pattern,
        case=False,
        na=False,
        regex=True
    )
)

nsf_clean_df["refined_relevant"] = (
    nsf_clean_df["in_project_date_scope"]
    & nsf_clean_df["strong_ai_match"]
    & nsf_clean_df["strong_chemistry_materials_match"]
)


nsf_original_candidate_count = (
    nsf_clean_df["baseline_relevant"].sum()
)

nsf_refined_candidate_count = (
    nsf_clean_df["refined_relevant"].sum()
)

nsf_removed_candidate_count = (
    nsf_clean_df["baseline_relevant"]
    & ~nsf_clean_df["refined_relevant"]
).sum()


print(
    "Original baseline candidates:",
    f"{nsf_original_candidate_count:,}"
)

print(
    "Refined candidates:",
    f"{nsf_refined_candidate_count:,}"
)

print(
    "Baseline candidates removed by refinement:",
    f"{nsf_removed_candidate_count:,}"
)


display(
    nsf_clean_df.loc[
        nsf_relevance_review_sample_df.index,
        [
            "award_id_clean",
            "title_clean",
            "strong_ai_match",
            "strong_chemistry_materials_match",
            "refined_relevant"
        ]
    ]
)

Original baseline candidates: 2,624
Refined candidates: 1,757
Baseline candidates removed by refinement: 907


,award_id_clean,title_clean,strong_ai_match,strong_chemistry_materials_match,refined_relevant
1372,2119172,DMREF: Engineering the On-The-Fly Control of 3...,True,True,True
9459,2344344,NSF Convergence Accelerator Track L: An Integr...,True,False,False
6152,2407565,Collaborative Research: Studying Nearby Cosmic...,True,False,False
2383,2141660,CAREER: Integrative Pathway Analysis for Cance...,True,False,False
12032,2347727,ERI: SoundEYE: AI-Driven Sensory Augmentation ...,True,False,False
3902,2306092,An engineered platform to establish the role o...,True,False,False
13665,2118180,Collaborative Research: CyberTraining: Impleme...,True,True,True
229,2047034,Career: Correct-by-Learning Methods for Reliab...,True,False,False
470,2132672,I-Corps: Developing predictive computational m...,True,True,True
3417,2222074,Organotypic whole hemisphere models to probe s...,True,False,False


## Assigning NSF relevance tiers

The broad and refined filters are retained together rather than using a single pass/fail classification.

The comparison showed that:

- 1,717 awards matched both filters;
- 907 awards matched only the broad filter;
- 40 awards matched only the refined filter.

The refined-only awards were captured because the stricter terminology includes specific phrases such as `materials design`, `materials simulation`, and `materials informatics` that were not included in the original broad keyword list. Manual inspection confirmed that these awards were generally relevant.

The final relevance tiers are therefore assigned as follows:

- `core_match`: awards that pass the refined AI and chemistry/materials filter;
- `broad_match`: awards that pass the broad filter but not the refined filter;
- `out_of_scope`: awards that pass neither relevance rule.

This tiered approach preserves higher-recall borderline records without treating them as equally reliable as the core candidate set.

In [31]:
nsf_both_filter_count = (
    nsf_clean_df["baseline_relevant"]
    & nsf_clean_df["refined_relevant"]
).sum()

nsf_baseline_only_count = (
    nsf_clean_df["baseline_relevant"]
    & ~nsf_clean_df["refined_relevant"]
).sum()

nsf_refined_only_count = (
    nsf_clean_df["refined_relevant"]
    & ~nsf_clean_df["baseline_relevant"]
).sum()


nsf_clean_df["relevance_tier"] = "out_of_scope"

nsf_clean_df.loc[
    nsf_clean_df["baseline_relevant"],
    "relevance_tier"
] = "broad_match"

nsf_clean_df.loc[
    nsf_clean_df["refined_relevant"],
    "relevance_tier"
] = "core_match"


nsf_relevance_tier_counts_df = (
    nsf_clean_df["relevance_tier"]
    .value_counts()
    .rename_axis("relevance_tier")
    .reset_index(name="award_count")
)

display(nsf_relevance_tier_counts_df)

print(
    "Awards matching both filters:",
    f"{nsf_both_filter_count:,}"
)

print(
    "Broad-filter-only awards:",
    f"{nsf_baseline_only_count:,}"
)

print(
    "Refined-filter-only awards:",
    f"{nsf_refined_only_count:,}"
)

,relevance_tier,award_count
0,out_of_scope,17252
1,core_match,1757
2,broad_match,907


Awards matching both filters: 1,717
Broad-filter-only awards: 907
Refined-filter-only awards: 40


In [32]:
def extract_case_sensitive_matches(text, pattern):
    if pd.isna(text):
        return ""

    matches = {
        match.group(0)
        for match in re.finditer(
            pattern,
            str(text)
        )
    }

    return " | ".join(sorted(matches))


nsf_refined_only_df = (
    nsf_clean_df.loc[
        nsf_clean_df["refined_relevant"]
        & ~nsf_clean_df["baseline_relevant"]
    ]
    .copy()
)

nsf_refined_only_df["strong_ai_matches"] = (
    nsf_refined_only_df["relevance_text_core"]
    .apply(
        lambda text: extract_keyword_matches(
            text,
            nsf_strong_ai_pattern
        )
    )
)

nsf_refined_only_df["ai_acronym_matches"] = (
    nsf_refined_only_df["relevance_text_core"]
    .apply(
        lambda text: extract_case_sensitive_matches(
            text,
            r"\bAI(?:[- ](?:driven|enabled|based|assisted|powered))?\b"
        )
    )
)

nsf_refined_only_df["strong_chemistry_matches"] = (
    nsf_refined_only_df["relevance_text_core"]
    .apply(
        lambda text: extract_keyword_matches(
            text,
            nsf_strong_chemistry_materials_pattern
        )
    )
)

nsf_refined_only_df["abstract_preview"] = (
    nsf_refined_only_df["abstract_clean"]
    .str.slice(0, 300)
)

print(
    "Refined-only awards:",
    f"{len(nsf_refined_only_df):,}"
)

display(
    nsf_refined_only_df.sample(
        n=min(20, len(nsf_refined_only_df)),
        random_state=42
    )[
        [
            "award_id_clean",
            "award_year",
            "title_clean",
            "programme_name",
            "strong_ai_matches",
            "ai_acronym_matches",
            "strong_chemistry_matches",
            "abstract_preview"
        ]
    ]
)

Refined-only awards: 40


,award_id_clean,award_year,title_clean,programme_name,strong_ai_matches,ai_acronym_matches,strong_chemistry_matches,abstract_preview
4418,2322377,2023,CC* Data Storage: FASTER Data Infrastructure t...,Campus Cyberinfrastructure,artificial intelligence | machine learning,,materials design,Researchers using computational modeling and d...
3899,2237039,2023,CAREER: Using Physics-Based Machine Learning t...,"Mechanics of Materials and Str, CAREER: FACULT...",machine learning,,materials design,Metal fracture occurs when cracks or other fla...
3680,2325413,2023,Workshop: Data Driven and Computational Modeli...,Mechanics of Materials and Str,machine learning,,materials design | materials modeling,This award provides registration and travel su...
6705,2503861,2025,Workshop on Future Trends in Research and Educ...,"Mechanics of Materials and Str, AM-Advanced Ma...",artificial intelligence | machine learning,AI,materials design,This grant provides funding for travel support...
678,2046551,2021,CAREER: Topological Assessment in Granular Mat...,CONDENSED MATTER PHYSICS,machine learning,AI,materials design,Non-technical Abstract: Pouring a bucket of sa...
2956,2212419,2022,Collaborative Research: III: Medium: Condition...,Info Integration & Informatics,artificial intelligence | machine learning | r...,AI,materials design,Measuring the difference between probability d...
12091,2412395,2024,Collaborative Research: Metal Additive Manufac...,AM-Advanced Manufacturing,machine learning | neural networks,,materials design,Compared to traditional manufacturing processe...
6806,2512845,2025,Travel Support for 18th US National Congress o...,Mechanics of Materials and Str,artificial intelligence | scientific machine l...,,materials design,This grant provides funding for graduate stude...
17210,2142164,2022,CAREER: A Multichannel Convolutional Neural Ne...,"Mechanics of Materials and Str, GVF - Global V...",machine learning | neural network,,materials design,This Faculty Early Career Development (CAREER)...
1153,2053929,2021,Collaborative Research: AI-Driven Multi-Scale ...,EDSE-Engineering Design and Sy,artificial intelligence | machine learning,AI,materials design,The objective of this project is to improve th...


In [33]:
nsf_broad_only_review_df = (
    nsf_clean_df.loc[
        nsf_clean_df["baseline_relevant"]
        & ~nsf_clean_df["refined_relevant"]
    ]
    .sample(
        n=20,
        random_state=42
    )
    .copy()
)

nsf_broad_only_review_df["ai_matches"] = (
    nsf_broad_only_review_df["relevance_text_core"]
    .apply(
        lambda text: extract_keyword_matches(
            text,
            nsf_ai_pattern
        )
    )
)

nsf_broad_only_review_df["chemistry_materials_matches"] = (
    nsf_broad_only_review_df["relevance_text_core"]
    .apply(
        lambda text: extract_keyword_matches(
            text,
            nsf_chemistry_materials_pattern
        )
    )
)

nsf_broad_only_review_df["abstract_preview"] = (
    nsf_broad_only_review_df["abstract_clean"]
    .str.slice(0, 350)
)

display(
    nsf_broad_only_review_df[
        [
            "award_id_clean",
            "award_year",
            "title_clean",
            "programme_name",
            "ai_matches",
            "chemistry_materials_matches",
            "abstract_preview"
        ]
    ]
)

,award_id_clean,award_year,title_clean,programme_name,ai_matches,chemistry_materials_matches,abstract_preview
18210,2235857,2023,Collaborative Research: Exploring the Role of ...,"Mechanics of Materials and Str, Special Initia...",data-driven,polymer | synthesis,"In nature, fibrous materials such as biologica..."
6658,2449371,2025,I-Corps: Translation Potential of a Handheld S...,I-Corps,machine learning,molecules | spectroscopy,This I-Corps project is focused on the develop...
5188,2345655,2024,PFI-RP: Resilient and Energy-Efficient Memory ...,PFI-Partnrships for Innovation,ai | artificial intelligence | machine learnin...,synthesis,The broader impact of this Partnerships for In...
11609,2145736,2022,CAREER: Learning Mechanisms from Single Cell M...,"Cross-BIO Activities, Innovation: Bioinformatics",deep learning,molecular,This award is funded in whole or in part under...
12578,2019674,2021,STC: Center for Research On Programmable Plant...,STCs - 2021 Class,data-driven | predictive modeling,chemical | molecular,This NSF Science and Technology Center aims to...
15441,2301919,2023,ERI: The impact of ionizable lipid chemistry a...,Special Initiatives,data-driven,chemical | chemistry | molecular,Lipid nanoparticles are ultrasmall drug delive...
7656,2522656,2025,Collaborative Research: DMREF: NSF-DST: Metast...,"OFFICE OF MULTIDISCIPLINARY AC, DMREF, GVF - G...",ai | data-driven | machine learning,chemical | synthesis,This Designing Materials to Revolutionize and ...
4305,2322713,2023,SHF: Small: Explainable Machine Learning for B...,Software & Hardware Foundation,ai | artificial intelligence | machine learning,synthesis,"With the advance of chip technology, Computer-..."
19296,2522667,2025,Collaborative Research: DMREF: NSF-BSF: Moire-...,"GVF - Global Venture Fund, OFFICE OF MULTIDISC...",ai,spectroscopy | synthesis,Non-technical description: Twisted oxide heter...
7873,2120858,2021,I-Corps: Network-based artificial intelligence...,I-Corps,ai | artificial intelligence | neural network,molecular,The broader impact/commercial potential of thi...


## Distinguishing research awards from supporting award activities

Not every relevant NSF award represents a standard research project.

Manual review identified several awards that support the wider research and innovation ecosystem, including:

- conferences, workshops, symposia, and travel support;
- commercialisation programmes such as I-Corps, SBIR, STTR, and PFI;
- research infrastructure, equipment, facilities, and cyberinfrastructure.

These records remain relevant to the funding landscape and are therefore not removed.

Instead, an `award_activity_type` field is created to distinguish:

- `research`;
- `commercialisation`;
- `conference_or_workshop`;
- `infrastructure`.

This classification allows GrantScopeAI to preserve the full funding landscape while enabling users to filter for direct research opportunities when needed. It does not change the relevance tier of an award.

In [34]:
# Flag award formats that may need different interpretation
title_for_type = nsf_clean_df["title_clean"].fillna("")

nsf_clean_df["is_conference_or_workshop"] = (
    title_for_type.str.contains(
        r"\b(?:conference|workshop|symposium|travel support|student support)\b",
        case=False,
        na=False,
        regex=True
    )
)

nsf_clean_df["is_commercialisation_award"] = (
    title_for_type.str.contains(
        r"\b(?:I-Corps|SBIR|STTR|PFI-RP|PFI-TT)\b",
        case=False,
        na=False,
        regex=True
    )
)

nsf_clean_df["is_infrastructure_award"] = (
    title_for_type.str.contains(
        r"\b(?:research infrastructure|data infrastructure|"
        r"cyberinfrastructure|equipment|facility|facilities)\b",
        case=False,
        na=False,
        regex=True
    )
)

nsf_clean_df["award_activity_type"] = "research"

nsf_clean_df.loc[
    nsf_clean_df["is_infrastructure_award"],
    "award_activity_type"
] = "infrastructure"

nsf_clean_df.loc[
    nsf_clean_df["is_commercialisation_award"],
    "award_activity_type"
] = "commercialisation"

nsf_clean_df.loc[
    nsf_clean_df["is_conference_or_workshop"],
    "award_activity_type"
] = "conference_or_workshop"


activity_type_summary_df = (
    nsf_clean_df.loc[
        nsf_clean_df["relevance_tier"].isin(
            ["core_match", "broad_match"]
        )
    ]
    .groupby(
        ["relevance_tier", "award_activity_type"]
    )
    .size()
    .reset_index(name="award_count")
    .sort_values(
        ["relevance_tier", "award_count"],
        ascending=[True, False]
    )
)

display(activity_type_summary_df)

,relevance_tier,award_activity_type,award_count
3,broad_match,research,797
0,broad_match,commercialisation,70
1,broad_match,conference_or_workshop,31
2,broad_match,infrastructure,9
7,core_match,research,1573
5,core_match,conference_or_workshop,82
4,core_match,commercialisation,66
6,core_match,infrastructure,36


In [35]:
relevance_tier_order = pd.CategoricalDtype(
    categories=[
        "core_match",
        "broad_match",
        "out_of_scope"
    ],
    ordered=True
)

nsf_clean_df["relevance_tier"] = (
    nsf_clean_df["relevance_tier"]
    .astype(relevance_tier_order)
)

nsf_candidate_awards_df = (
    nsf_clean_df.loc[
        nsf_clean_df["relevance_tier"].isin(
            ["core_match", "broad_match"]
        )
    ]
    .copy()
    .sort_values(
        [
            "relevance_tier",
            "award_year",
            "amount_native"
        ],
        ascending=[True, False, False]
    )
    .reset_index(drop=True)
)

print(
    "Total NSF candidate awards:",
    f"{len(nsf_candidate_awards_df):,}"
)

print(
    "Core matches:",
    f"{nsf_candidate_awards_df['relevance_tier'].eq('core_match').sum():,}"
)

print(
    "Broad matches:",
    f"{nsf_candidate_awards_df['relevance_tier'].eq('broad_match').sum():,}"
)

print(
    "Duplicate award IDs:",
    nsf_candidate_awards_df[
        "award_id_clean"
    ].duplicated().sum()
)

print(
    "Missing candidate titles:",
    nsf_candidate_awards_df[
        "title_clean"
    ].eq("").sum()
)

print(
    "Missing candidate source URLs:",
    nsf_candidate_awards_df[
        "source_url"
    ].isna().sum()
)

display(
    nsf_candidate_awards_df[
        [
            "award_id_clean",
            "award_year",
            "title_clean",
            "relevance_tier",
            "award_activity_type",
            "organisation_name",
            "amount_native"
        ]
    ].head(10)
)

Total NSF candidate awards: 2,664
Core matches: 1,757
Broad matches: 907
Duplicate award IDs: 0
Missing candidate titles: 0
Missing candidate source URLs: 0


,award_id_clean,award_year,title_clean,relevance_tier,award_activity_type,organisation_name,amount_native
0,2433348,2025,AI-Materials Institute (AI-MI),core_match,research,Cornell University,20000000.0
1,2445868,2025,"MIP: Biomaterials, Polymers, and Advanced Cons...",core_match,research,University of California-Santa Barbara,19800000.0
2,2503773,2025,Center: NSF Center for Synthetic Organic Elect...,core_match,research,Missouri University of Science and Technology,19800000.0
3,2503933,2025,NSF Center for Single-Entity Nanochemistry and...,core_match,research,Indiana University,19800000.0
4,2503885,2025,NSF Center for Genetically Encoded Materials,core_match,research,University of California-Berkeley,18000000.0
5,2505932,2025,"AI Institute for Molecular Discovery, Syntheti...",core_match,research,University of Illinois at Urbana-Champaign,15000000.0
6,2532263,2025,Entrepreneurial Fellowships to Enhance U.S. Co...,core_match,research,"ACTIVATE GLOBAL, INC.",14978985.0
7,2452693,2025,Ideas Lab: CFIRE: PRESENT: PRotein Evolution i...,core_match,research,"CARAVEL BIO, INC.",7800000.0
8,2453663,2025,Ideas Lab: CFIRE: Electricity-Driven Cell-Free...,core_match,research,ARZEDA Corp.,7786412.0
9,2514731,2025,EPSCoR CREST Phase I: Center for Energy Techno...,core_match,research,Montana Technological University,7500000.0


In [36]:
nsf_final_validation_df = pd.DataFrame({
    "check": [
        "Full cleaned award rows",
        "Unique award IDs",
        "Duplicate award IDs",
        "Unique grant keys",
        "Candidate awards",
        "Core matches",
        "Broad matches",
        "Candidate duplicate IDs",
        "Missing candidate titles",
        "Missing candidate abstracts",
        "Missing candidate amounts",
        "Missing candidate organisations",
        "Missing candidate source URLs",
        "Non-USD candidate records"
    ],
    "result": [
        len(nsf_clean_df),
        nsf_clean_df["award_id_clean"].nunique(),
        nsf_clean_df["award_id_clean"].duplicated().sum(),
        nsf_clean_df["grant_key"].nunique(),
        len(nsf_candidate_awards_df),
        nsf_candidate_awards_df[
            "relevance_tier"
        ].eq("core_match").sum(),
        nsf_candidate_awards_df[
            "relevance_tier"
        ].eq("broad_match").sum(),
        nsf_candidate_awards_df[
            "award_id_clean"
        ].duplicated().sum(),
        nsf_candidate_awards_df[
            "title_clean"
        ].eq("").sum(),
        nsf_candidate_awards_df[
            "abstract_clean"
        ].eq("").sum(),
        nsf_candidate_awards_df[
            "amount_native"
        ].isna().sum(),
        nsf_candidate_awards_df[
            "organisation_name"
        ].isna().sum(),
        nsf_candidate_awards_df[
            "source_url"
        ].isna().sum(),
        nsf_candidate_awards_df[
            "currency"
        ].ne("USD").sum()
    ]
})

display(nsf_final_validation_df)

print(
    "Candidate award years:",
    sorted(
        nsf_candidate_awards_df[
            "award_year"
        ].dropna().unique().tolist()
    )
)

print(
    "Candidate activity types:",
    sorted(
        nsf_candidate_awards_df[
            "award_activity_type"
        ].dropna().unique().tolist()
    )
)

,check,result
0,Full cleaned award rows,19916
1,Unique award IDs,19916
2,Duplicate award IDs,0
3,Unique grant keys,19916
4,Candidate awards,2664
5,Core matches,1757
6,Broad matches,907
7,Candidate duplicate IDs,0
8,Missing candidate titles,0
9,Missing candidate abstracts,0


Candidate award years: [2021, 2022, 2023, 2024, 2025]
Candidate activity types: ['commercialisation', 'conference_or_workshop', 'infrastructure', 'research']


In [37]:
# Define the NSF processed-data output folder

nsf_processed_dir = (
    PROJECT_ROOT
    / "Data"
    / "Processed_Data"
    / "NSF"
)

nsf_processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("NSF processed-data directory:")
print(nsf_processed_dir)

NSF processed-data directory:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\NSF


In [38]:
# Create a compact, application-ready NSF candidate dataset

nsf_candidate_compact_columns = [
    "source",
    "grant_key",
    "award_id_clean",
    "title_clean",
    "abstract_clean",
    "startDate",
    "expDate",
    "award_year",
    "amount_native",
    "amount_obligated",
    "currency",
    "organisation_name",
    "organisation_city",
    "organisation_state_code",
    "organisation_country_code",
    "programme_name",
    "directorate_name",
    "division_name",
    "lead_investigator",
    "is_active_award",
    "source_url",
    "relevance_tier",
    "award_activity_type",
    "strong_ai_match",
    "strong_chemistry_materials_match",
    "matched_query_count",
    "has_query_overlap"
]

nsf_candidate_compact_df = (
    nsf_candidate_awards_df[
        nsf_candidate_compact_columns
    ]
    .copy()
    .rename(
        columns={
            "award_id_clean": "source_id",
            "title_clean": "title",
            "abstract_clean": "abstract",
            "startDate": "start_date",
            "expDate": "end_date"
        }
    )
)

nsf_compact_output_path = (
    nsf_processed_dir
    / "nsf_candidate_awards_compact_2021_2025.csv"
)

nsf_candidate_compact_df.to_csv(
    nsf_compact_output_path,
    index=False,
    encoding="utf-8-sig"
)

print(
    "Compact candidate rows:",
    f"{len(nsf_candidate_compact_df):,}"
)

print(
    "Compact candidate columns:",
    len(nsf_candidate_compact_df.columns)
)

print(
    "Duplicate grant keys:",
    nsf_candidate_compact_df[
        "grant_key"
    ].duplicated().sum()
)

print("\nCompact candidate dataset:")
print(nsf_compact_output_path)

print(
    "File size:",
    f"{nsf_compact_output_path.stat().st_size / 1_000_000:.2f} MB"
)

display(
    nsf_candidate_compact_df.head()
)

Compact candidate rows: 2,664
Compact candidate columns: 27
Duplicate grant keys: 0

Compact candidate dataset:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\NSF\nsf_candidate_awards_compact_2021_2025.csv
File size: 9.56 MB


,source,grant_key,source_id,title,abstract,start_date,end_date,award_year,amount_native,amount_obligated,...,division_name,lead_investigator,is_active_award,source_url,relevance_tier,award_activity_type,strong_ai_match,strong_chemistry_materials_match,matched_query_count,has_query_overlap
0,NSF,NSF_2433348,2433348,AI-Materials Institute (AI-MI),The need for materials with improved or new pr...,2025-10-01,2030-09-30,2025,20000000.0,6000000.0,...,Division Of Materials Research,Eun-Ah Kim,True,https://www.nsf.gov/awardsearch/showAward?AWD_...,core_match,research,True,True,2,True
1,NSF,NSF_2445868,2445868,"MIP: Biomaterials, Polymers, and Advanced Cons...","The BioPACIFIC MIP (Biomaterials, Polymers and...",2025-08-01,2030-07-31,2025,19800000.0,5800000.0,...,Division Of Materials Research,Javier R Read de Alaniz,True,https://www.nsf.gov/awardsearch/showAward?AWD_...,core_match,research,True,True,4,True
2,NSF,NSF_2503773,2503773,Center: NSF Center for Synthetic Organic Elect...,The ability to use electrons as chemical reage...,2025-09-01,2030-08-31,2025,19800000.0,3800000.0,...,Division Of Chemistry,Shelley D Minteer,True,https://www.nsf.gov/awardsearch/showAward?AWD_...,core_match,research,True,True,2,True
3,NSF,NSF_2503933,2503933,NSF Center for Single-Entity Nanochemistry and...,The NSF Center for Single-Entity Nanochemistry...,2025-09-01,2030-08-31,2025,19800000.0,3800000.0,...,Division Of Chemistry,Sara E Skrabalak,True,https://www.nsf.gov/awardsearch/showAward?AWD_...,core_match,research,True,True,4,True
4,NSF,NSF_2503885,2503885,NSF Center for Genetically Encoded Materials,The Center for Genetically Encoded Materials (...,2025-09-01,2030-08-31,2025,18000000.0,3000000.0,...,Division Of Chemistry,Alanna Schepartz,True,https://www.nsf.gov/awardsearch/showAward?AWD_...,core_match,research,True,True,2,True


In [39]:
from pathlib import Path

nsf_processed_dir = (
    Path(PROJECT_ROOT)
    / "Data"
    / "Processed_Data"
    / "NSF"
)

nsf_processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

nsf_full_output_path = (
    nsf_processed_dir
    / "nsf_awards_cleaned_2021_2025.csv"
)

nsf_candidate_output_path = (
    nsf_processed_dir
    / "nsf_candidate_awards_2021_2025.csv"
)

nsf_validation_output_path = (
    nsf_processed_dir
    / "nsf_cleaning_validation_summary.csv"
)

nsf_clean_df.to_csv(
    nsf_full_output_path,
    index=False,
    encoding="utf-8-sig"
)

nsf_candidate_awards_df.to_csv(
    nsf_candidate_output_path,
    index=False,
    encoding="utf-8-sig"
)

nsf_final_validation_df.to_csv(
    nsf_validation_output_path,
    index=False,
    encoding="utf-8-sig"
)

for label, path in {
    "Full cleaned dataset": nsf_full_output_path,
    "Candidate dataset": nsf_candidate_output_path,
    "Validation summary": nsf_validation_output_path
}.items():

    print(f"\n{label}:")
    print(path)
    print(
        "File size:",
        f"{path.stat().st_size / 1_000_000:.2f} MB"
    )


Full cleaned dataset:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\NSF\nsf_awards_cleaned_2021_2025.csv
File size: 433.85 MB

Candidate dataset:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\NSF\nsf_candidate_awards_2021_2025.csv
File size: 60.23 MB

Validation summary:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\NSF\nsf_cleaning_validation_summary.csv
File size: 0.00 MB


In [40]:
# Exclude large NSF files while keeping the compact candidate dataset

gitignore_path = PROJECT_ROOT / ".gitignore"

nsf_gitignore_entries = [
    "",
    "# NSF raw and large processed datasets",
    "Data/Raw_Data/nsf_grantscope_raw_*.csv",
    "Data/Processed_Data/NSF/nsf_awards_cleaned_*.csv",
    "Data/Processed_Data/NSF/nsf_candidate_awards_2021_2025.csv"
]

if gitignore_path.exists():
    existing_gitignore_text = gitignore_path.read_text(
        encoding="utf-8"
    )
else:
    existing_gitignore_text = ""

entries_added = []

with gitignore_path.open(
    "a",
    encoding="utf-8"
) as gitignore_file:

    for entry in nsf_gitignore_entries:
        if entry and entry not in existing_gitignore_text:
            gitignore_file.write(f"{entry}\n")
            entries_added.append(entry)

print(".gitignore path:")
print(gitignore_path)

print("\nEntries added:")
if entries_added:
    for entry in entries_added:
        print(f"- {entry}")
else:
    print("No new entries were needed.")

print("\nCompact candidate file remains available for GitHub:")
print(nsf_compact_output_path)

.gitignore path:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\.gitignore

Entries added:
No new entries were needed.

Compact candidate file remains available for GitHub:
C:\Users\kahau\OneDrive\Documents\Techmeup\Final_Project\GrantScopeAI\Data\Processed_Data\NSF\nsf_candidate_awards_compact_2021_2025.csv


## NSF cleaning and preparation summary

The NSF Awards dataset has been cleaned, consolidated, classified, and exported for use in GrantScopeAI.

### Final dataset

- 33,125 raw API rows were consolidated into 19,916 unique NSF awards.
- Duplicate rows represented repeated query retrievals rather than separate grants.
- Query lineage was preserved without allowing extraction queries to influence relevance classification.
- All retained awards fall within the 2021–2025 project period.
- `estimatedTotalAmt` was selected as the primary award value, while obligated funding was retained separately.
- Programme, organisation, investigator, location, status, and source-link fields were standardized.

### Relevance classification

Awards were classified using a two-stage AI and chemistry/materials keyword methodology:

- 1,757 awards were classified as `core_match`;
- 907 awards were classified as `broad_match`;
- 17,252 awards were retained in the cleaned dataset as `out_of_scope`.

The final candidate dataset therefore contains 2,664 unique NSF awards.

The broad tier is retained because manual review showed that it contains both borderline-relevant projects and incidental keyword matches. This allows GrantScopeAI to prioritize higher-confidence awards while preserving wider discovery coverage.

### Award activity types

Candidate awards were additionally labelled as:

- direct research;
- commercialisation;
- conference or workshop support;
- research infrastructure.

These labels do not change relevance scores but allow users to distinguish standard research grants from supporting funding activities.

### Outputs

Three main processed outputs were created:

- a full cleaned NSF dataset for local analysis;
- a detailed candidate dataset for local validation;
- a compact candidate dataset for GitHub, Streamlit, Tableau, and cross-source integration.

The large full and diagnostic files are excluded through `.gitignore`, while the compact 2,664-award dataset remains available for version control.